In [1]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
import torch.nn.functional as F
import warnings 
from torch_geometric.nn import SAGEConv
from gensim.models import Word2Vec
from tqdm import tqdm
import math
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

/home/cs/grad/nasrm1/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/usr/lib/python3/dist-packages/paramiko/transport.py:261: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,


In [2]:
def prepare_graph(df):
    nodes, labels, edges = {}, {}, []
    dummies = {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7}

    for _, row in df.iterrows():
        actor_id, object_id = row["actorID"], row["objectID"]
        action = row["action"]

        for entity_id in [actor_id, object_id]:
            nodes.setdefault(entity_id, []).append(action)
            if entity_id == actor_id:
                labels[entity_id] = dummies[row['actor_type']]
            else:
                labels[entity_id] = dummies[row['object']]

        edges.append((actor_id, object_id))

    features, feat_labels, edge_index, mapping = [], [], [[], []], []
    index_map = {}

    for key, value in nodes.items():
        index_map[key] = len(features)
        features.append(value)
        feat_labels.append(labels[key])
        mapping.append(key)

    for source, target in edges:
        edge_index[0].append(index_map[source])
        edge_index[1].append(index_map[target])

    return features, feat_labels, edge_index, mapping

In [3]:
class GCN(torch.nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.conv1 = SAGEConv(in_channel, 32, normalize=True)
        self.conv2 = SAGEConv(32, out_channel, normalize=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return x
    
model = GCN(30,8).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)


In [4]:
class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]

def infer(document):
    word_embeddings = [w2vmodel.wv[word] for word in document if word in  w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(20)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(30)
w2vmodel = Word2Vec.load("./trained_weights/streamspot/streamspot.model")


# Before Evasion

In [5]:
thresh = 200 
correct_benign = 0
correct_attack = 0

In [6]:
model.load_state_dict(torch.load(f'trained_weights/streamspot/lstreamspot.pth',map_location=torch.device('cpu')))
model.eval()

GCN(
  (conv1): SAGEConv(30, 32)
  (conv2): SAGEConv(32, 8)
)

In [13]:
import numpy as np
import pandas as pd
import random
from sklearn.metrics.pairwise import cosine_similarity

number_of_nodes = []
avg_similarities = []

for i in tqdm(range(300, 400)):
    with open(f"./streamspot_dependancies/streamspot/{i}.txt") as f:
        data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame(data, columns=['actorID', 'actor_type', 'objectID', 'object', 'action', 'timestamp'])
    df = df.dropna()

    phrases, labels, edges, mapp = prepare_graph(df)

    number_of_nodes.append(len(phrases))

    # Filter nodes with label 0
    zero_label_indices = [idx for idx, lbl in enumerate(labels) if lbl == 0]
    
    if not zero_label_indices:
        avg_similarities.append(np.nan)
        continue

    # Infer embeddings for label=0 nodes
    zero_label_nodes = [infer(phrases[idx]) for idx in zero_label_indices]
    zero_label_nodes = np.array(zero_label_nodes)

    # Randomly sample up to 1000 nodes
    if len(zero_label_nodes) > 1000:
        sampled_nodes = zero_label_nodes[random.sample(range(len(zero_label_nodes)), 1000)]
    else:
        sampled_nodes = zero_label_nodes

    # Compute cosine similarity matrix
    sim_matrix = cosine_similarity(sampled_nodes)

    # Take upper triangle (excluding diagonal) and compute mean similarity
    triu_indices = np.triu_indices_from(sim_matrix, k=1)
    avg_sim = np.mean(sim_matrix[triu_indices])

    avg_similarities.append(avg_sim)

print("Average number of nodes per graph:", np.mean(number_of_nodes))
print("Average cosine similarity across graphs:", np.nanmean(avg_similarities))


100%|██████████| 100/100 [01:06<00:00,  1.50it/s]

Average number of nodes per graph: 8890.8
Average cosine similarity across graphs: 0.6677608


In [11]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import random

number_of_nodes = []
cross_label_similarities = []

for i in tqdm(range(300, 400)):
    with open(f"./streamspot_dependancies/streamspot/{i}.txt") as f:
        data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame(data, columns=['actorID', 'actor_type', 'objectID', 'object', 'action', 'timestamp'])
    df = df.dropna()
    
    phrases, labels, edges, mapp = prepare_graph(df)

    number_of_nodes.append(len(phrases))

    # Get indices for each label
    zero_indices = [idx for idx, lbl in enumerate(labels) if lbl == 0]
    one_indices  = [idx for idx, lbl in enumerate(labels) if lbl == 1]

    # Need at least 1 node from each label to compute cross-label similarity
    if zero_indices and one_indices:
        # Infer embeddings
        zero_nodes = np.array([infer(phrases[idx]) for idx in zero_indices])
        one_nodes  = np.array([infer(phrases[idx]) for idx in one_indices])

        # Sample up to 1000 from each label
        if len(zero_nodes) > 100:
            zero_nodes = zero_nodes[random.sample(range(len(zero_nodes)), 100)]
        if len(one_nodes) > 100:
            one_nodes = one_nodes[random.sample(range(len(one_nodes)), 100)]

        # Compute pairwise cosine similarity between label 0 and label 1 nodes
        sim_matrix = cosine_similarity(zero_nodes, one_nodes)
        avg_cross_sim = np.mean(sim_matrix)
        cross_label_similarities.append(avg_cross_sim)

print("Average number of nodes per graph:", np.mean(number_of_nodes))
print("Average cosine similarity between label=0 and label=1 nodes:", np.mean(cross_label_similarities))


100%|██████████| 100/100 [01:06<00:00,  1.50it/s]

Average number of nodes per graph: 8890.8
Average cosine similarity between label=0 and label=1 nodes: 0.0075364923


In [9]:
for i in range(450,600):
    f = open(f"./streamspot_dependancies/streamspot/{i}.txt")
    data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
    df = df.dropna()
    
    phrases,labels,edges,mapp = prepare_graph(df)

    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)
    
    graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))
    graph.n_id = torch.arange(graph.num_nodes)
    flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool)

    out = model(graph.x, graph.edge_index)

    sorted, indices = out.sort(dim=1,descending=True)
    conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
    conf = (conf - conf.min()) / conf.max()

    pred = indices[:,0]
    cond = ~(pred == graph.y)

    if cond.sum() <= thresh:
         correct_benign = correct_benign + 1
    
    # print(cond.sum().item(), (cond.sum().item() / len(cond))*100)
print("Number Correctly classified benign samples: ", correct_benign)

Number Correctly classified benign samples:  150


In [10]:
for i in range(300,400):
    f = open(f"./streamspot_dependancies/streamspot/{i}.txt")
    data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
    df = df.dropna()
    
    phrases,labels,edges,mapp = prepare_graph(df)
  
    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)
    
    graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))
    graph.n_id = torch.arange(graph.num_nodes)
    flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool)

    out = model(graph.x, graph.edge_index)

    sorted, indices = out.sort(dim=1,descending=True)
    conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
    conf = (conf - conf.min()) / conf.max()

    pred = indices[:,0]
    cond = ~(pred == graph.y)

    if cond.sum() > thresh:
         correct_attack = correct_attack + 1
            
    # print(cond.sum().item(), (cond.sum().item() / len(cond))*100)
print("Number Correctly classified attack samples: ", correct_attack)

Number Correctly classified attack samples:  95


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import Data

# Make sure this folder exists
os.makedirs("ss_flagged_indices", exist_ok=True)

correct_attack = 0
thresh = 200  # set your threshold value appropriately

for i in range(300, 400):
    with open(f"./streamspot_dependancies/streamspot/{i}.txt") as f:
        data = f.read().split('\n')

    data = [line.split('\t') for line in data if line.strip() != '']
    df = pd.DataFrame(data, columns=['actorID', 'actor_type', 'objectID', 'object', 'action', 'timestamp'])
    df = df.dropna()

    phrases, labels, edges, mapp = prepare_graph(df)

    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)

    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float).to(device),
        y=torch.tensor(labels, dtype=torch.long).to(device),
        edge_index=torch.tensor(edges, dtype=torch.long).to(device)
    )
    graph.n_id = torch.arange(graph.num_nodes)
    flag = torch.tensor([True] * graph.num_nodes, dtype=torch.bool)

    out = model(graph.x, graph.edge_index)

    sorted, indices = out.sort(dim=1, descending=True)
    conf = (sorted[:, 0] - sorted[:, 1]) / sorted[:, 0]
    conf = (conf - conf.min()) / conf.max()

    pred = indices[:, 0]
    cond = ~(pred == graph.y)  # flagged nodes

    if cond.sum() > thresh:
        correct_attack += 1

    # Save flagged node indices
    flagged_indices = torch.where(cond)[0].tolist()
    output_path = f"ss_flagged_indices/indices_{i}.txt"
    with open(output_path, 'w') as fout:
        for idx in flagged_indices:
            fout.write(f"{idx}\n")

print("Number Correctly classified attack samples: ", correct_attack)


Number Correctly classified attack samples:  100


In [7]:
import os
import pandas as pd

# Ensure output directory exists
os.makedirs("ss_flagged_IDs", exist_ok=True)

for i in range(300, 400):
    # 1. Load raw graph data
    try:
        with open(f"./streamspot_dependancies/streamspot/{i}.txt") as f:
            data = f.read().split('\n')
    except FileNotFoundError:
        continue  # skip missing graph files

    data = [line.split('\t') for line in data if line.strip()]
    df = pd.DataFrame(data, columns=['actorID', 'actor_type', 'objectID', 'object', 'action', 'timestamp'])
    df = df.dropna()

    # 2. Get mapping from index to original node ID
    _, _, _, mapp = prepare_graph(df)

    # 3. Load flagged indices
    indices_path = f"ss_flagged_indices/indices_{i}.txt"
    if not os.path.exists(indices_path):
        continue  # skip if no flagged indices found

    with open(indices_path) as f:
        flagged_indices = [int(line.strip()) for line in f if line.strip().isdigit()]

    # 4. Convert indices to system entity IDs using mapp
    flagged_ids = [mapp[idx] for idx in flagged_indices if idx < len(mapp)]

    # 5. Save the IDs to new file
    output_path = f"ss_flagged_IDs/ids_{i}.txt"
    with open(output_path, 'w') as f:
        for node_id in flagged_ids:
            f.write(f"{node_id}\n")


In [ ]:
TOTAL_ATTACKS = 100
TOTAL_BENIGN = 150

def calculate_metrics(correct_attack, correct_benign):
    TP = correct_attack
    FP = TOTAL_BENIGN - correct_benign
    TN = correct_benign
    FN = TOTAL_ATTACKS - correct_attack

    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
    TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
    TNR = TN / (TN + FP) if (TN + FP) > 0 else 0

    print(f"Number of True Positives: {TP}")
    print(f"Number of False Positives: {FP}")
    print(f"Number of False Negatives: {FN}")
    print(f"Number of True Negatives: {TN}\n")

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")

    fscore = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    print(f"Fscore: {fscore}\n")

    print("False Positive Rate (FPR):", FPR)
    print("True Positive Rate (TPR):", TPR)
    print("True Negative Rate (TNR):", TNR)

calculate_metrics(correct_attack, correct_benign)


Number of True Positives: 0
Number of False Positives: 0
Number of False Negatives: 100
Number of True Negatives: 150

Precision: 0
Recall: 0.0
Fscore: 0

False Positive Rate (FPR): 0.0
True Positive Rate (TPR): 0.0
True Negative Rate (TNR): 1.0


# After Evasion

In [5]:
thresh = 200
correct_benign = 150
correct_attack = 0

In [6]:
model.load_state_dict(torch.load('./trained_weights/streamspot/lstreamspot.pth', map_location=device))
model.eval()

GCN(
  (conv1): SAGEConv(30, 32)
  (conv2): SAGEConv(32, 8)
)

In [ ]:
for i in tqdm(range(450, 600)):
    f = open(f"./streamspot_dependancies/streamspot/{i}.txt")
    data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
    df = df.dropna()
    
    phrases,labels,edges,mapp = prepare_graph(df)

    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)
    
    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float, device=device),
        y=torch.tensor(labels, dtype=torch.long, device=device),
        edge_index=torch.tensor(edges, dtype=torch.long, device=device)
    )
    graph.n_id = torch.arange(graph.num_nodes, device=device)
    flag = torch.ones(graph.num_nodes, dtype=torch.bool, device=device)

    out = model(graph.x, graph.edge_index)

    sorted, indices = out.sort(dim=1,descending=True)
    conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
    conf = (conf - conf.min()) / conf.max()

    pred = indices[:,0]
    cond = ~(pred == graph.y)

    if cond.sum() <= thresh:
         correct_benign = correct_benign + 1
    
#     print(cond.sum().item(), (cond.sum().item() / len(cond))*100)
print("Number of correct benign predictions: ", correct_benign)

In [7]:
def split_malicious_nodes_by_label(malicious_indices, labels):

    malicious_groups = defaultdict(list)

    for node_idx in malicious_indices:
        node_label = labels[node_idx]
        malicious_groups[node_label].append(node_idx)

    return dict(malicious_groups)

def split_nodes_by_label_exclude_malicious(phrases, labels, indices_of_malicious_nodes):
    node_groups = defaultdict(list)
    malicious_set = set(indices_of_malicious_nodes)  # Convert to set for faster lookup

    for node_idx, node_features in enumerate(phrases):
        # Check if the node index is in the malicious nodes list
        if node_idx not in malicious_set:
            node_label = labels[node_idx]
            node_groups[node_label].append(node_idx)

    return dict(node_groups)

def filter_nodes_by_phrase_length(node_groups, phrases, min_len=20, max_len=40):

    filtered_groups = {}

    for label, node_indices in node_groups.items():
        filtered_groups[label] = [
            idx for idx in node_indices if min_len <= len(phrases[idx]) <= max_len
        ]

    return filtered_groups

def compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes):
    top_nodes_by_similarity = {}

    for label in malicious_groups.keys():
        if label in filtered_nodes_grouped:
            malicious_indices = malicious_groups[label]
            filtered_indices = filtered_nodes_grouped[label]

            # Get feature vectors for malicious and filtered nodes
            malicious_vectors = np.array([nodes[idx] for idx in malicious_indices])
            filtered_vectors = np.array([nodes[idx] for idx in filtered_indices])

            # Compute cosine similarity (malicious nodes vs filtered nodes)
            similarities = cosine_similarity(filtered_vectors, malicious_vectors)  # Shape: (filtered, malicious)

            # Take the max similarity score for each filtered node
            max_similarities = similarities.max(axis=1)  # Max similarity per filtered node

            # Step 1: Use np.argsort to get the indices that would sort the max_similarities array
            sorted_indices = np.array(filtered_indices)[np.argsort(-max_similarities)]  # Negative for descending order

            # Step 2: Select most similar nodes for the current label
            top_nodes_by_similarity[label] = sorted_indices 

    return top_nodes_by_similarity


def duplicate_and_replace(df, target_id, new_id):
    # Find rows where target_id appears in actorID or objectID
    mask = df['actorID'].isin([target_id]) | df['objectID'].isin([target_id])
    selected_rows = df[mask]

    # Replace target_id with new_id in actorID and objectID using replace() method
    selected_rows[['actorID', 'objectID']] = selected_rows[['actorID', 'objectID']].replace(target_id, new_id)

    # Return the modified rows, no need to append to df within the function
    return selected_rows

In [ ]:
number_of_triggred_actions = []
number_of_flagged_nodes = []
all_lines = []

correct_attack = 0
thresh = 200 

for i in tqdm(range(300,400)):

     graph_number = i

     f = open(f"./streamspot_dependancies/streamspot/{i}.txt")
     data = f.read().split('\n')

     data = [line.split('\t') for line in data]
     df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
     df = df.dropna()
    
     phrases,labels,edges,mapp = prepare_graph(df)
     
     nodes = [infer(x) for x in phrases]
     nodes = np.array(nodes)
    
     graph = Data(
          x=torch.tensor(nodes, dtype=torch.float).to(device),
          y=torch.tensor(labels, dtype=torch.long).to(device), 
          edge_index=torch.tensor(edges, dtype=torch.long).to(device)
     )
    
     graph.n_id = torch.arange(graph.num_nodes, device=device)

     flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool, device=device)

     out = model(graph.x, graph.edge_index)

     sorted, indices = out.sort(dim=1,descending=True)
     conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
     conf = (conf - conf.min()) / conf.max()

     pred = indices[:,0]
     cond = ~(pred == graph.y)

     # Flag nodes where predictions are incorrect
     wrong_indices = torch.nonzero(cond).squeeze().cpu().numpy()

     flagged_indices = list(wrong_indices)
     
     ############################################################################################################
     # EVASION
     ############################################################################################################
     
     # Step 1: Benign Node Selection
     malicious_groups = split_malicious_nodes_by_label(flagged_indices, labels) 

     # Split all nodes by their labels excluding the malicious nodes
     all_nodes_grouped_excluding_malicious = split_nodes_by_label_exclude_malicious(phrases, labels, flagged_indices)

     # Step 2: Minimal Interaction Selection
     filtered_nodes_grouped = filter_nodes_by_phrase_length(all_nodes_grouped_excluding_malicious, phrases, 20, 40)

     # Step 3: Contextual Consistency Filtering, Compute top similar nodes by similarity
     top_nodes_by_similarity = compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes)


     # Prepare a list to collect the modified rows
     modified_rows = []

     # Precompute all malicious node IDs and their most similar benign node IDs in advance
     malicious_and_benign = {}

     for i in flagged_indices:
          malicious_node_ID = mapp[i]  # Malicious node ID from the mapping
          malicious_label = labels[i]  # Get the malicious node label
          
          try:
               # Get all similar benign node IDs for the malicious label, limit to 10
               benign_candidates = top_nodes_by_similarity[malicious_label]
               # benign_candidates = filtered_nodes_grouped[malicious_label]
               if len(benign_candidates) > 25:
                    benign_candidates = benign_candidates[:25]  # Limit to top 25

               similar_benign_node_IDs = [mapp[node] for node in benign_candidates]
          except KeyError:
               continue
          
          # Store the mapping of malicious node to all similar benign node IDs
          malicious_and_benign[malicious_node_ID] = similar_benign_node_IDs

     # Use tqdm for the longest loop
     for malicious_node_ID, benign_node_IDs in malicious_and_benign.items():
          for benign_node_ID in benign_node_IDs:  # Iterate over all similar benign nodes (max 10)
               selected_rows = duplicate_and_replace(df, benign_node_ID, malicious_node_ID)
               modified_rows.append(selected_rows)

     # After all iterations, concatenate all the new rows once
     df_curated = pd.concat([df] + modified_rows, ignore_index=True)

     # print("graph_number:",graph_number)

     cur_lines = df_curated.apply(
          lambda row: f"{row['actorID']}\t{row['actor_type']}\t{row['objectID']}\t{row['object']}\t{row['action']}\t{graph_number}\n",
          axis=1
     ).tolist()

     all_lines.extend(cur_lines)  # add to global list

     phrases,labels,edges,mapp = prepare_graph(df_curated)

     number_of_triggred_actions.append(len(modified_rows))

     nodes = [infer(x) for x in phrases]
     nodes = np.array(nodes)
    
     graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))
     graph.n_id = torch.arange(graph.num_nodes)
     flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool)

     out = model(graph.x, graph.edge_index)

     sorted, indices = out.sort(dim=1,descending=True)
     conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
     conf = (conf - conf.min()) / conf.max()

     pred = indices[:,0]
     cond = ~(pred == graph.y)

     if cond.sum() > thresh:
          correct_attack = correct_attack + 1

     number_of_flagged_nodes.append(cond.sum().item())

print("Number of correct attack predictions: ", correct_attack)
print("Number of triggered actions: ", sum(number_of_triggred_actions))
print("Average number of triggered actions: ", sum(number_of_triggred_actions)/len(number_of_triggred_actions))
print("Average number of flagged nodes: ", sum(number_of_flagged_nodes)/len(number_of_flagged_nodes))
print("Average number of flagged nodes: ", sum(number_of_flagged_nodes))
print("Max number of flagged nodes: ", max(number_of_flagged_nodes))
print("Min number of flagged nodes: ", min(number_of_flagged_nodes))


100%|██████████| 100/100 [14:06<00:00,  8.47s/it]


Number of correct attack predictions:  0
Number of triggered actions:  303246
Average number of triggered actions:  3032.46
Average number of flagged nodes:  58.43
Average number of flagged nodes:  5843
Max number of flagged nodes:  101
Min number of flagged nodes:  26


In [ ]:
TOTAL_ATTACKS = 100
TOTAL_BENIGN = 150

def calculate_metrics(correct_attack, correct_benign):
    TP = correct_attack
    FP = TOTAL_BENIGN - correct_benign
    TN = correct_benign
    FN = TOTAL_ATTACKS - correct_attack

    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
    TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
    TNR = TN / (TN + FP) if (TN + FP) > 0 else 0

    print(f"Number of True Positives: {TP}")
    print(f"Number of False Positives: {FP}")
    print(f"Number of False Negatives: {FN}")
    print(f"Number of True Negatives: {TN}\n")

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")

    fscore = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    print(f"Fscore: {fscore}\n")

    print("False Positive Rate (FPR):", FPR)
    print("True Positive Rate (TPR):", TPR)
    print("True Negative Rate (TNR):", TNR)

calculate_metrics(correct_attack, correct_benign)


Number of True Positives: 0
Number of False Positives: 0
Number of False Negatives: 100
Number of True Negatives: 150

Precision: 0
Recall: 0.0
Fscore: 0

False Positive Rate (FPR): 0.0
True Positive Rate (TPR): 0.0
True Negative Rate (TNR): 1.0


In [8]:
from tqdm import tqdm
import pandas as pd

total_entities = 0
total_events = 0
file_count = 0

# Process files from 300 to 399
for i in tqdm(list(range(300, 400)) + list(range(450, 600)), desc="Processing Graphs"):
    with open(f"./streamspot_dependancies/streamspot/{i}.txt") as f:
        data = f.read().split('\n')

    data = [line.split('\t') for line in data if line.strip() != '']
    df = pd.DataFrame(data, columns=['actorID', 'actor_type', 'objectID', 'object', 'action', 'timestamp'])
    df = df.dropna()

    number_of_sys_entities = len(set(df['actorID']).union(set(df['objectID'])))
    number_of_events = len(df)

    total_entities += number_of_sys_entities
    total_events += number_of_events
    file_count += 1

average_entities = total_entities / file_count
average_events = total_events / file_count

print(f"Processed {file_count} graphs.")
print(f"Total system entities across all graphs: {total_entities}")
print(f"Average system entities per graph: {average_entities:.2f}")
print(f"Total events across all graphs: {total_events}")
print(f"Average events per graph: {average_events:.2f}")


Processing Graphs: 100%|██████████| 250/250 [04:02<00:00,  1.03it/s]

Processed 250 graphs.
Total system entities across all graphs: 2231273
Average system entities per graph: 8925.09
Total events across all graphs: 46499507
Average events per graph: 185998.03
